In [1]:
from langchain_teddynote.korean import stopwords
from langchain_teddynote import logging
from dotenv import load_dotenv

logging.langsmith("CH09-VectorStores")
load_dotenv()


LangSmith 추적을 시작합니다.
[프로젝트명]
CH09-VectorStores


False

In [2]:
stop_words = stopwords()
stop_words[:20]

['아',
 '휴',
 '아이구',
 '아이쿠',
 '아이고',
 '어',
 '나',
 '우리',
 '저희',
 '따라',
 '의해',
 '을',
 '를',
 '에',
 '의',
 '가',
 '으로',
 '로',
 '에게',
 '뿐이다']

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
import glob

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
split_docs = []

files=sorted(glob.glob("data/*.pdf"))

for file in files:
    loader = PyMuPDFLoader(file)
    split_docs.extend(loader.load_and_split(text_splitter))
    
len(split_docs)

C:\Users\user\AppData\Local\Temp\ipykernel_22300\124605300.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
d:\rag_one\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


119

In [4]:
split_docs[0].metadata

{'producer': 'Hancom PDF 1.3.0.542',
 'creator': 'Hwp 2018 10.0.0.13462',
 'creationdate': '2023-12-08T13:28:38+09:00',
 'source': 'data\\SPRI_AI_Brief_2023년12월호_F.pdf',
 'file_path': 'data\\SPRI_AI_Brief_2023년12월호_F.pdf',
 'total_pages': 23,
 'format': 'PDF 1.4',
 'title': '',
 'author': 'dj',
 'subject': '',
 'keywords': '',
 'moddate': '2023-12-08T13:28:38+09:00',
 'trapped': '',
 'modDate': "D:20231208132838+09'00'",
 'creationDate': "D:20231208132838+09'00'",
 'page': 0}

In [5]:
from langchain_teddynote.community.pinecone import preprocess_documents

contents, metadatas = preprocess_documents(
    split_docs=split_docs,
    metadata_keys = ["source","page","author"],
    min_length=5,
    use_basename=True,
)

100%|██████████| 119/119 [00:00<?, ?it/s]


In [6]:
contents[:5]

['2023년 12월호',
 '2023년 12월호\nⅠ. 인공지능 산업 동향 브리프\n 1. 정책/법제 \n   ▹ 미국, 안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령 발표  ························· 1\n   ▹ G7, 히로시마 AI 프로세스를 통해 AI 기업 대상 국제 행동강령에 합의··························· 2\n   ▹ 영국 AI 안전성 정상회의에 참가한 28개국, AI 위험에 공동 대응 선언··························· 3',
 '▹ 미국 법원, 예술가들이 생성 AI 기업에 제기한 저작권 소송 기각····································· 4\n   ▹ 미국 연방거래위원회, 저작권청에 소비자 보호와 경쟁 측면의 AI 의견서 제출················· 5\n   ▹ EU AI 법 3자 협상, 기반모델 규제 관련 견해차로 난항··················································· 6\n \n 2. 기업/산업',
 '2. 기업/산업 \n   ▹ 미국 프런티어 모델 포럼, 1,000만 달러 규모의 AI 안전 기금 조성································ 7\n   ▹ 코히어, 데이터 투명성 확보를 위한 데이터 출처 탐색기 공개  ······································· 8\n   ▹ 알리바바 클라우드, 최신 LLM ‘통이치엔원 2.0’ 공개 ······················································ 9',
 '▹ 삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개 ··························································· 10\n   ▹ 구글, 앤스로픽에 20억 달러 투자로 생성 AI 협력 강화 ···········································

In [7]:
metadatas["source"][:5]

['SPRI_AI_Brief_2023년12월호_F.pdf',
 'SPRI_AI_Brief_2023년12월호_F.pdf',
 'SPRI_AI_Brief_2023년12월호_F.pdf',
 'SPRI_AI_Brief_2023년12월호_F.pdf',
 'SPRI_AI_Brief_2023년12월호_F.pdf']

In [8]:
len(contents), len(metadatas["source"]), len(metadatas["page"])

(119, 119, 119)

In [9]:
import os
from langchain_teddynote.community.pinecone import create_index

pc_index = create_index(
    api_key=os.environ["PINECONE_API_KEY"],
    index_name="isaac-db-index",
    dimension=4096,
    metric="dotproduct", #euclidean, cosine
)

[create_index]
DescribeIndexStatsResponse(dimension=4096, total_vector_count=119, metric='dotproduct', namespaces=1)


In [10]:
from langchain_teddynote.community.pinecone import(
    create_sparse_encoder,
    fit_sparse_encoder,
)
sparse_encoder = create_sparse_encoder(stop_words,mode="kiwi")

saved_path = fit_sparse_encoder(
    sparse_encoder=sparse_encoder,
    contents=contents,
    save_path="./sparse_encoder.pkl"
)

100%|██████████| 119/119 [00:01<00:00, 85.72it/s]

[fit_sparse_encoder]
Saved Sparse Encoder to: ./sparse_encoder.pkl


In [11]:
from langchain_teddynote.community.pinecone import load_sparse_encoder

sparse_encoder = load_sparse_encoder("./sparse_encoder.pkl")

[load_sparse_encoder]
Loaded Sparse Encoder from: ./sparse_encoder.pkl


In [12]:
from langchain_openai import OpenAIEmbeddings
from langchain_upstage import UpstageEmbeddings

openai_embeddings = OpenAIEmbeddings(model="text-embeddings-3-large")
upstage_embeddings = UpstageEmbeddings(model="solar-embedding-1-large-passage")


In [13]:
%%time
from langchain_teddynote.community.pinecone import upsert_documents
from langchain_upstage import UpstageEmbeddings

upsert_documents(
    index=pc_index,
    namespace="isaac-study-namespace-01",
    contents=contents,
    metadatas=metadatas,
    sparse_encoder=sparse_encoder,
    embedder=upstage_embeddings,
    batch_size=32
)

100%|██████████| 4/4 [00:12<00:00,  3.14s/it]


[upsert_documents]
DescribeIndexStatsResponse(dimension=4096, total_vector_count=238, metric='dotproduct', namespaces=1)
CPU times: total: 2.41 s
Wall time: 12.8 s


In [14]:
%%time
from langchain_teddynote.community.pinecone import upsert_documents_parallel

upsert_documents_parallel(
    index=pc_index,
    namespace="isaac-study-namespace-2",
    contents=contents,
    metadatas=metadatas,
    sparse_encoder=sparse_encoder,
    embedder=upstage_embeddings,
    batch_size=64,
    max_workers=30
)

문서 Upsert 중:   0%|          | 0/2 [00:00<?, ?it/s]

문서 Upsert 중: 100%|██████████| 2/2 [00:03<00:00,  1.99s/it]


총 119개의 Vector 가 Upsert 되었습니다.
DescribeIndexStatsResponse(dimension=4096, total_vector_count=357, metric='dotproduct', namespaces=2)
CPU times: total: 1 s
Wall time: 4.27 s


In [15]:
pc_index.describe_index_stats()

DescribeIndexStatsResponse(dimension=4096, total_vector_count=357, metric='dotproduct', namespaces=2)

In [18]:
from langchain_teddynote.community.pinecone import delete_namespace
delete_namespace(
    pinecone_index=pc_index,
    namespace="isaac-study-namespace-01"
)

네임스페이스 'isaac-study-namespace-01'의 모든 데이터가 삭제되었습니다.


In [19]:
pc_index.describe_index_stats()

DescribeIndexStatsResponse(dimension=4096, total_vector_count=119, metric='dotproduct', namespaces=1)

In [21]:
import os

from langchain_teddynote.korean import stopwords
from langchain_teddynote.community.pinecone import init_pinecone_index
from langchain_upstage import UpstageEmbeddings

pinecone_params = init_pinecone_index(
    index_name="isaac-db-index",
    namespace="isaac-study-namespace-2",
    api_key=os.environ["PINECONE_API_KEY"],
    sparse_encoder_path="./sparse_encoder.pkl",
    stopwords=stopwords(),
    tokenizer="kiwi",
    embeddings=UpstageEmbeddings(
        model="solar-embedding-1-large-query"
    ),
    top_k=5,
    alpha=0.5,    
)

[init_pinecone_index]
DescribeIndexStatsResponse(dimension=4096, total_vector_count=119, metric='dotproduct', namespaces=1)


In [22]:
from langchain_teddynote.community.pinecone import PineconeKiwiHybridRetriever
pinecone_retriever = PineconeKiwiHybridRetriever(**pinecone_params)

In [24]:
search_results = pinecone_retriever.invoke("gpt-4o 미니 출시 관련 정보에 대해서 알려줘")
for result in search_results:
    print(result.page_content)
    print(result.metadata)
    print("\n","==========================="*5,"\n")

생성에서 가장 우수한 성능을 발휘
KEY Contents
£ 주요 LLM 중 GPT-4가 가장 환각 현상 적고 GPT-3.5 터보도 비슷한 성능 기록
n 머신러닝 데이터 관리 기업 갈릴레오(Galileo)가 2023년 11월 15일 주요 LLM의 환각 현상을 평가한 
‘LLM 환각 지수(LLM Hallucination Index)’를 발표
∙생성 AI의 환각 현상은 AI 시스템이 잘못된 정보를 생성하거나, 현실과 다른 부정확한 결과를 내놓는
{'author': 'dj', 'context': '생성에서 가장 우수한 성능을 발휘\nKEY Contents\n£ 주요 LLM 중 GPT-4가 가장 환각 현상 적고 GPT-3.5 터보도 비슷한 성능 기록\nn 머신러닝 데이터 관리 기업 갈릴레오(Galileo)가 2023년 11월 15일 주요 LLM의 환각 현상을 평가한 \n‘LLM 환각 지수(LLM Hallucination Index)’를 발표\n∙생성 AI의 환각 현상은 AI 시스템이 잘못된 정보를 생성하거나, 현실과 다른 부정확한 결과를 내놓는', 'page': 19.0, 'source': 'SPRI_AI_Brief_2023년12월호_F.pdf'}


▹ 구글 딥마인드, 범용 AI 모델의 기능과 동작에 대한 분류 체계 발표······························ 16
   ▹ 갈릴레오의 LLM 환각 지수 평가에서 GPT-4가 가장 우수 ··········································· 17
   
 4. 인력/교육     
   ▹ 영국 옥스퍼드 인터넷 연구소, AI 기술자의 임금이 평균 21% 높아······························· 18
   
   
 
Ⅱ. 주요 행사
{'author': 'dj', 'context': '▹ 구글 딥마인드, 범용 AI 모델의 기능과 동작에 대한 분류 체계 발표······························ 16\n   ▹ 갈릴레오의 LLM 환각 지수 평가에서 G

In [25]:
search_results = pinecone_retriever.invoke("gpt-4o 미니 출시 관련 정보에 대해서 알려줘", search_kwargs={"k":1})
for result in search_results:
    print(result.page_content)
    print(result.metadata)
    print("\n","==========================="*5,"\n")

생성에서 가장 우수한 성능을 발휘
KEY Contents
£ 주요 LLM 중 GPT-4가 가장 환각 현상 적고 GPT-3.5 터보도 비슷한 성능 기록
n 머신러닝 데이터 관리 기업 갈릴레오(Galileo)가 2023년 11월 15일 주요 LLM의 환각 현상을 평가한 
‘LLM 환각 지수(LLM Hallucination Index)’를 발표
∙생성 AI의 환각 현상은 AI 시스템이 잘못된 정보를 생성하거나, 현실과 다른 부정확한 결과를 내놓는
{'author': 'dj', 'context': '생성에서 가장 우수한 성능을 발휘\nKEY Contents\n£ 주요 LLM 중 GPT-4가 가장 환각 현상 적고 GPT-3.5 터보도 비슷한 성능 기록\nn 머신러닝 데이터 관리 기업 갈릴레오(Galileo)가 2023년 11월 15일 주요 LLM의 환각 현상을 평가한 \n‘LLM 환각 지수(LLM Hallucination Index)’를 발표\n∙생성 AI의 환각 현상은 AI 시스템이 잘못된 정보를 생성하거나, 현실과 다른 부정확한 결과를 내놓는', 'page': 19.0, 'source': 'SPRI_AI_Brief_2023년12월호_F.pdf'}




In [30]:
search_results = pinecone_retriever.invoke("앤스로픽", search_kwargs={"alpha":1,"k":3})
# 여기서 alpha 옵션은 밀집 벡터를 의미하고 1인 경우 밀집벡터 1.0 희소벡터 0.0이됩니다. 
# 다시 말해 유사도만 측정하여 의미 검색에 초점을 맞추게됩니다. 
# 희소 벡터의 값이 높아지면 키워드가 검색어에 포함될 가능성이 높아집니다.
for result in search_results:
    print(result.page_content)
    print(result.metadata)
    print("\n","==========================="*5,"\n")

£ 구글, 앤스로픽에 최대 20억 달러 투자 합의 및 클라우드 서비스 제공
n 구글이 2023년 10월 27일 앤스로픽에 최대 20억 달러를 투자하기로 합의했으며, 이 중 5억 
달러를 우선 투자하고 향후 15억 달러를 추가로 투자할 방침
∙구글은 2023년 2월 앤스로픽에 이미 5억 5,000만 달러를 투자한 바 있으며, 아마존도 지난 9월 
앤스로픽에 최대 40억 달러의 투자 계획을 공개
∙한편, 2023년 11월 8일 블룸버그 보도에 따르면 앤스로픽은 구글의 클라우드 서비스 사용을 위해
{'author': 'dj', 'context': '£ 구글, 앤스로픽에 최대 20억 달러 투자 합의 및 클라우드 서비스 제공\nn 구글이 2023년 10월 27일 앤스로픽에 최대 20억 달러를 투자하기로 합의했으며, 이 중 5억 \n달러를 우선 투자하고 향후 15억 달러를 추가로 투자할 방침\n∙구글은 2023년 2월 앤스로픽에 이미 5억 5,000만 달러를 투자한 바 있으며, 아마존도 지난 9월 \n앤스로픽에 최대 40억 달러의 투자 계획을 공개\n∙한편, 2023년 11월 8일 블룸버그 보도에 따르면 앤스로픽은 구글의 클라우드 서비스 사용을 위해', 'page': 13.0, 'source': 'SPRI_AI_Brief_2023년12월호_F.pdf'}


1. 정책/법제  
2. 기업/산업 
3. 기술/연구 
 4. 인력/교육
구글, 앤스로픽에 20억 달러 투자로 생성 AI 협력 강화 
n 구글이 앤스로픽에 최대 20억 달러 투자에 합의하고 5억 달러를 우선 투자했으며, 앤스로픽은 
구글과 클라우드 서비스 사용 계약도 체결
n 3대 클라우드 사업자인 구글, 마이크로소프트, 아마존은 차세대 AI 모델의 대표 기업인 
앤스로픽 및 오픈AI와 협력을 확대하는 추세
KEY Contents
£ 구글, 앤스로픽에 최대 20억 달러 투자 합의 및 클라우드 서비스 제공
{'author': 'dj', 'context': '1. 정책/법제  \n2. 기업/산업 \n3. 기술/연구 \n 

In [31]:
search_results = pinecone_retriever.invoke("앤스로픽", search_kwargs={"alpha":0,"k":3})
# 여기서 alpha 옵션은 밀집 벡터를 의미하고 0인 경우 밀집벡터 0.0 희소벡터 1.0이됩니다. 
# 다시 말해 유사도만 측정하여 의미 검색에 초점을 맞추게됩니다. 
# 희소 벡터의 값이 높아지면 키워드가 검색어에 포함될 가능성이 높아집니다.
for result in search_results:
    print(result.page_content)
    print(result.metadata)
    print("\n","==========================="*5,"\n")

1. 정책/법제  
2. 기업/산업 
3. 기술/연구 
 4. 인력/교육
구글, 앤스로픽에 20억 달러 투자로 생성 AI 협력 강화 
n 구글이 앤스로픽에 최대 20억 달러 투자에 합의하고 5억 달러를 우선 투자했으며, 앤스로픽은 
구글과 클라우드 서비스 사용 계약도 체결
n 3대 클라우드 사업자인 구글, 마이크로소프트, 아마존은 차세대 AI 모델의 대표 기업인 
앤스로픽 및 오픈AI와 협력을 확대하는 추세
KEY Contents
£ 구글, 앤스로픽에 최대 20억 달러 투자 합의 및 클라우드 서비스 제공
{'author': 'dj', 'context': '1. 정책/법제  \n2. 기업/산업 \n3. 기술/연구 \n 4. 인력/교육\n구글, 앤스로픽에 20억 달러 투자로 생성 AI 협력 강화 \nn 구글이 앤스로픽에 최대 20억 달러 투자에 합의하고 5억 달러를 우선 투자했으며, 앤스로픽은 \n구글과 클라우드 서비스 사용 계약도 체결\nn 3대 클라우드 사업자인 구글, 마이크로소프트, 아마존은 차세대 AI 모델의 대표 기업인 \n앤스로픽 및 오픈AI와 협력을 확대하는 추세\nKEY Contents\n£ 구글, 앤스로픽에 최대 20억 달러 투자 합의 및 클라우드 서비스 제공', 'page': 13.0, 'source': 'SPRI_AI_Brief_2023년12월호_F.pdf'}


£ 구글, 앤스로픽에 최대 20억 달러 투자 합의 및 클라우드 서비스 제공
n 구글이 2023년 10월 27일 앤스로픽에 최대 20억 달러를 투자하기로 합의했으며, 이 중 5억 
달러를 우선 투자하고 향후 15억 달러를 추가로 투자할 방침
∙구글은 2023년 2월 앤스로픽에 이미 5억 5,000만 달러를 투자한 바 있으며, 아마존도 지난 9월 
앤스로픽에 최대 40억 달러의 투자 계획을 공개
∙한편, 2023년 11월 8일 블룸버그 보도에 따르면 앤스로픽은 구글의 클라우드 서비스 사용을 위해
{'author': 'dj', 'context': '£ 구글, 앤스로픽에 최대 20억 달러 투